In [48]:
from dlfs.model import TransformerDecoderModel

from dlfs.loss import CCE_Loss
from dlfs.optimizers import Optimizer_Adam

import numpy as np

# Tiny Shakespeare dataset

In [49]:
with open('../../data/input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(f'Number of characters in the whole text: {len(text)}')

Number of characters in the whole text: 1115394


In [50]:
print(f'{text[:200]}')

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


In [51]:
chars = sorted(list(set(text))) # get sorted unique characters
vocab_size = len(chars) # number of unique characters
print(f'All characters: {"".join(chars)}')
print(f'Vocab size: {vocab_size}')

All characters: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
Vocab size: 65


# Character level tokenizer

In [52]:
encoded_dict = {chars[i]: i for i in range(vocab_size)} # map character to int
decoded_dict = {i: chars[i] for i in range(vocab_size)} # map int to character

def encode(s: str) -> list[int]:
    return [encoded_dict[c] for c in s]

def decode(s: list[int]) -> str:
    return "".join([decoded_dict[c] for c in s])

print(encode("test string"))
print(decode(encode("test string")))

[58, 43, 57, 58, 1, 57, 58, 56, 47, 52, 45]
test string


# Convert whole dataset to indices

In [53]:
data = encode(text)
data = np.array(data, dtype=np.int32)
print(data.shape)

(1115394,)


# Train test split

In [54]:
n = int(0.9*len(data))
X_train, X_test = data[:n], data[n:]
print(f'n: {n}\nX_train: {X_train.shape}\nX_test: {X_test.shape}')

n: 1003854
X_train: (1003854,)
X_test: (111540,)


# Creating sequences

In [55]:
def create_sequences(data, seq_len = 8):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i+seq_len])
        y.append(data[i+1:i+seq_len+1])
    return np.array(X), np.array(y)

seq_len = 64

X_train, y_train = create_sequences(X_train, seq_len)
print(f'{X_train.shape}, {y_train.shape}')
print(f'There are {X_train.shape[0]} sequences, each sequence is of length {X_train.shape[1]}')

(1003790, 64), (1003790, 64)
There are 1003790 sequences, each sequence is of length 64


# Example input sequence and target sequence

In [56]:
print(X_train[0, :5]) # first couple of tokens
print(y_train[0, :5])

[18 47 56 57 58]
[47 56 57 58  1]


# Model training

In [25]:
np.random.seed(1337)

batch_size = 64

n_embed = 192
n_head = 6
n_dec_layers = 3
dim_ff = 768
dropout = 0.2
eps = 1e-5

epochs = 5000
lr = 3e-4

loss = CCE_Loss(from_logits=True)
optimizer = Optimizer_Adam(learning_rate=lr, decay=0., clip_grad=False)

model = TransformerDecoderModel(vocab_size=vocab_size, 
                                seq_len=seq_len, 
                                n_embed=n_embed, 
                                n_head=n_head,
                                n_dec_layers=n_dec_layers,
                                dim_ff=dim_ff, 
                                dropout=dropout,
                                layer_norm_eps=eps,
                                loss_function=loss, 
                                optimizer=optimizer)

model.train(X_train, y_train, print_every=1, epochs=epochs, batch_size=batch_size)

Vocab size: 65, block size: 64
===== EPOCH : 0 ===== LOSS : 5.337087961037117 =====
===== EPOCH : 1 ===== LOSS : 4.861753767425042 =====
===== EPOCH : 2 ===== LOSS : 4.485652664228953 =====
===== EPOCH : 3 ===== LOSS : 4.263953252581425 =====
===== EPOCH : 4 ===== LOSS : 4.092121147880681 =====
===== EPOCH : 5 ===== LOSS : 3.986379572093691 =====
===== EPOCH : 6 ===== LOSS : 3.894072491897912 =====
===== EPOCH : 7 ===== LOSS : 3.83958133008292 =====
===== EPOCH : 8 ===== LOSS : 3.80487872106316 =====
===== EPOCH : 9 ===== LOSS : 3.7640932651719714 =====
===== EPOCH : 10 ===== LOSS : 3.7016089379576615 =====
===== EPOCH : 11 ===== LOSS : 3.6988111620191764 =====
===== EPOCH : 12 ===== LOSS : 3.661244756650274 =====
===== EPOCH : 13 ===== LOSS : 3.679303289905304 =====
===== EPOCH : 14 ===== LOSS : 3.63513764478676 =====
===== EPOCH : 15 ===== LOSS : 3.6513579904444797 =====
===== EPOCH : 16 ===== LOSS : 3.6439326977917323 =====
===== EPOCH : 17 ===== LOSS : 3.6174619231973297 =====
====

KeyboardInterrupt: 

In [28]:
print(decode(model.generate(np.zeros((1, 1), dtype=np.int32), seq_len, max_new_tokens=500)[0].tolist()))


lxoenye p e.tsFarnn;Uo almgrrG gasermyryt em tn    $d  m,t em!t:bmityss e  ouMo.osotau  :xy u uvaenEfh d  tteoCE gtnl  au  toew hist
oo reR tejt nt r iactsSttbm h ufmtyoq   .ne:tyI   
 fo
lose nu  n h oo.tmwo bwios tiwRa
io' n
n?a .oeu duuss oXh oo tleyfweah srmlh r hntrT:tfHeahley&h tewe  shlarntH p mmwtoAh e b
etat  annoAoeeafts$t ih
afryst,, iu
dy:iotut  obtto tcy i ;icoot hdeUli  hT 
idmaod sho dtl sa,v.nhtMw, seL uer Iedt UrtA hn  o  
n r t! !rltTmo h.u uogox tO,  onmAeop.  ot uho mc ypi it
